# Demo — EDDOps Golden Dataset Validation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kpassoubady/bedrock-companion/blob/main/day1/demos/demo-eddops-validation/demo-eddops-validation.ipynb)

This notebook is a follow-along demo.

Day 1 — Block 6: Best Practices and Wrap-Up

Demonstrates Evaluation-Driven Development (EDDOps): gating agent
promotion from DRAFT to PUBLISHED using a golden dataset. Shows
how evaluation evidence prevents regression in agent behavior.

In [ ]:
# Install required dependencies
!pip install boto3 --quiet

In [ ]:
import json

def print_section(title: str):
    print(f"\n{'=' * 60}")
    print(f"  {title}")
    print(f"{'=' * 60}")

def show_eddops_pipeline():
    """Explain the EDDOps promotion pipeline.

    Agent promotion through the Registry (DRAFT → REVIEW → PUBLISHED)
    must be gated by evaluation evidence against golden datasets."""
    print_section("EDDOps — Evaluation-Driven Development Pipeline")

    pipeline = {
        "stages": [
            {
                "stage": "DRAFT",
                "description": "Agent under active development",
                "gate": "Code review + unit tests",
            },
            {
                "stage": "REVIEW",
                "description": "Agent awaiting evaluation",
                "gate": "Golden dataset evaluation (≥95% pass rate)",
            },
            {
                "stage": "PUBLISHED",
                "description": "Agent serving production traffic",
                "gate": "Canary deployment + monitoring period",
            },
        ],
        "rollback": "Alias update to previous PUBLISHED version",
    }

    print(f"  {json.dumps(pipeline, indent=2)}")
    print()
    print("  Key principle: Agent promotion is gated by evidence, not trust.")
    print("  A code change that passes review but fails evaluation does NOT ship.")

def run_golden_dataset_evaluation(agent_version: str):
    """Simulate running a golden dataset evaluation.

    The golden dataset contains input/output pairs that define correct
    agent behavior. Any deviation is a regression."""
    print_section(f"Golden Dataset Evaluation — {agent_version}")

    golden_dataset = [
        {
            "input": "Book a flight to Seattle",
            "expected_tool": "book_flight",
            "expected_params": {"destination": "Seattle"},
        },
        {
            "input": "What is the weather in NYC?",
            "expected_tool": "check_weather",
            "expected_params": {"city": "NYC"},
        },
        {
            "input": "Cancel my subscription",
            "expected_tool": "human_escalation",
            "expected_params": {"reason": "user_requested_cancellation"},
        },
    ]

    results = []
    for item in golden_dataset:
        # Simulate agent execution
        actual_tool = item["expected_tool"]

        # Inject a regression for v2.0-draft
        if "Cancel" in item["input"] and agent_version == "v2.0-draft":
            actual_tool = "cancel_account"

        passed = actual_tool == item["expected_tool"]
        results.append(
            {
                "input": item["input"],
                "expected_tool": item["expected_tool"],
                "actual_tool": actual_tool,
                "passed": passed,
            }
        )

        status = "PASS" if passed else "FAIL"
        print(f"  [{status}] Input: '{item['input']}'")
        print(f"         Expected: {item['expected_tool']} → Actual: {actual_tool}")

    score = sum(1 for r in results if r["passed"]) / len(results)
    print(f"\n  Evaluation Score: {score * 100:.0f}%")

    if score >= 0.95:
        print("  Status: APPROVED for promotion to PUBLISHED")
    else:
        print("  Status: REJECTED — Fix regression and re-evaluate")

    return results

def show_production_best_practices():
    """Five production best practices for agent operations."""
    print_section("Five Production Best Practices")

    practices = [
        {
            "practice": "1. Continuous Evaluation (EDDOps)",
            "description": "Gate promotion on golden-dataset evidence, not code review alone",
            "implementation": "CI/CD stage runs evals → pass/fail gates promotion",
        },
        {
            "practice": "2. Deterministic Boundaries",
            "description": "Use IAM, tool allowlists, and network rules — not prompts — for security",
            "implementation": "IAM policies, Gateway allowlists, VPC egress rules",
        },
        {
            "practice": "3. Multi-Agent Validation",
            "description": "Validator agent reviews primary agent's output before execution",
            "implementation": "Step Functions orchestrate primary → validator → execute or escalate",
        },
        {
            "practice": "4. Human-in-the-Loop (HITL)",
            "description": "Human escalation is a designed workflow component, not a failure mode",
            "implementation": "Defined SLAs, approval UI, audit trail for human decisions",
        },
        {
            "practice": "5. Observability is Non-Negotiable",
            "description": "End-to-end tracing is mandatory for root-cause analysis",
            "implementation": "CloudWatch dashboards, alarms, anomaly detection, cost attribution",
        },
    ]

    for p in practices:
        print(f"\n  {p['practice']}")
        print(f"    {p['description']}")
        print(f"    → {p['implementation']}")

def main():
    print("EDDOps and Production Best Practices — Instructor Demo\n")
    show_eddops_pipeline()

    print("\n" + "=" * 60)
    print("  Scenario 1: Stable version (v1.0)")
    print("=" * 60)
    run_golden_dataset_evaluation("v1.0-draft")

    print("\n" + "=" * 60)
    print("  Scenario 2: Regression detected (v2.0)")
    print("=" * 60)
    run_golden_dataset_evaluation("v2.0-draft")

    show_production_best_practices()

    print_section("Key Takeaways")
    print("  1. EDDOps gates promotion on evaluation evidence, not trust")
    print("  2. Golden datasets define correct behavior — deviation = regression")
    print("  3. AgentCore Registry: DRAFT → REVIEW → PUBLISHED")
    print("  4. Five production practices span eval, security, validation, HITL, observability")
    print("  5. AgentCore provides infrastructure; you provide governance and business logic")

if __name__ == "__main__":
    main()
